# Wearable HRV and Sleep Efficiency Analysis

**Author:** Xuanyue Zhang  
**M.S. Biological Data Science, Arizona State University**

This portfolio notebook presents a concise version of my capstone analysis of real-world wearable heart rate variability (HRV) and sleep diary data.

The project evaluates whether HRV features can predict nightly sleep efficiency, compares pre-sleep and overnight analysis windows, and explores participant symptom profiles using clustering.


## 1. Research Questions

1. Can wearable-derived HRV features predict nightly sleep efficiency?
2. Does predictive performance differ between a **2-hour pre-sleep window** and an **overnight window**?
3. Do expanded HRV feature sets improve prediction beyond a smaller core feature set?
4. Are symptom profiles based on PHQ-9, GAD-7, and ISI associated with differences in sleep efficiency?


## 2. Data Source and Reproducibility

The analysis uses the public dataset described by Baigutanova et al. (2025), available from Figshare:

**DOI:** https://doi.org/10.6084/m9.figshare.28509740

The workflow combines:

- wearable HRV sensor data
- nightly sleep diary data
- participant-level symptom survey data

Raw source data are not redistributed in this repository. The code below is designed to reproduce the analysis after the source CSV files are downloaded and placed in the working directory.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans

from xgboost import XGBRegressor

RANDOM_STATE = 42


## 3. Data Quality Control

Wearable HRV data were quality-controlled before nightly aggregation.

The primary segment-level rule was:

**missingness_score ≤ 0.35**

Because some values in the original sensor file were negative, missingness scores were clipped to the valid range \([0,1]\) before applying the threshold.

Two analysis windows were evaluated:

- **Pre-sleep:** 2 hours before sleep onset
- **Overnight:** sleep onset to wake time


In [ ]:
# Example QC workflow
sensor = pd.read_csv("sensor_hrv.csv")

sensor["missingness_score"] = (
    sensor["missingness_score"]
    .clip(lower=0, upper=1)
)

sensor_qc = sensor.loc[
    sensor["missingness_score"] <= 0.35
].copy()

qc_pass_rate = len(sensor_qc) / len(sensor)
print(f"Segment-level QC pass rate: {qc_pass_rate:.1%}")


### Feasibility after QC

The original analysis showed substantially greater HRV availability overnight than during the 2-hour pre-sleep window.

| Window | Segment threshold | Retained nights before participant filter | Retained participants |
|---|---:|---:|---:|
| Pre-sleep | ≥5 valid segments | 212 | 35 |
| Overnight | ≥10 valid segments | 559 | 44 |

For participant-wise modeling, participants also needed multiple retained nights:

| Final modeling dataset | Nights | Participants | Median nights / participant |
|---|---:|---:|---:|
| Pre-sleep | 202 | 25 | 6.0 |
| Overnight | 554 | 40 | 13.5 |

This made the overnight window more feasible for stable model evaluation.


## 4. Feature Engineering

Night-level predictors were generated by aggregating HRV measurements within each retained window.

### Core HRV features
- RMSSD
- SDNN
- Heart rate

### Expanded HRV features
- SDSD
- pNN20
- pNN50
- LF
- HF
- LF/HF

For each HRV variable, nightly summary statistics such as **mean, median, and standard deviation** were calculated.


In [ ]:
# Example feature aggregation pattern
hrv_features = [
    "HR", "sdnn", "rmssd", "sdsd",
    "pnn20", "pnn50", "lf", "hf", "lf/hf"
]

agg_map = {
    feature: ["mean", "median", "std"]
    for feature in hrv_features
}

# window_df should contain one row per QC-passing 5-minute segment
# and identifiers such as deviceId and date.
nightly_features = (
    window_df
    .groupby(["deviceId", "date"])
    .agg(agg_map)
)

nightly_features.columns = [
    f"{feature}_{stat}"
    for feature, stat in nightly_features.columns
]

nightly_features = nightly_features.reset_index()


## 5. Modeling Strategy

The central modeling decision was to prevent participant leakage.

Because each participant contributed multiple nights, random row-level train/test splitting would allow nights from the same person to appear in both training and test sets. Instead, models were evaluated with **participant-wise GroupKFold cross-validation**, using `deviceId` as the grouping variable.

Models:

- **ElasticNetCV** — regularized linear model
- **XGBoost** — nonlinear tree-based model
- **Mean predictor baseline**

Metrics:

- MAE
- RMSE
- R²


In [ ]:
def evaluate_grouped_model(df, features, model, n_splits=5):
    X = df[features]
    y = df["sleep_efficiency"].astype(float).to_numpy()
    groups = df["deviceId"].to_numpy()

    splitter = GroupKFold(n_splits=n_splits)
    rows = []

    for fold, (train_idx, test_idx) in enumerate(
        splitter.split(X, y, groups), start=1
    ):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        rows.append({
            "fold": fold,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
            "R2": r2_score(y_test, pred),
        })

    return pd.DataFrame(rows)


elastic_net = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.9, 1.0],
        alphas=np.logspace(-4, 0, 50),
        cv=5,
        max_iter=20000
    ))
])

xgb = XGBRegressor(
    random_state=RANDOM_STATE,
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8
)


## 6. Primary Modeling Results

Participant-wise cross-validation showed that HRV-only models had limited ability to generalize to unseen participants.

### Core HRV feature set

| Window | Model | MAE (mean ± SD) | RMSE (mean ± SD) | R² (mean ± SD) |
|---|---|---:|---:|---:|
| Pre-sleep | Mean baseline | 0.0481 ± 0.0070 | 0.0710 ± 0.0167 | -0.0993 ± 0.1213 |
| Pre-sleep | ElasticNetCV | 0.0481 ± 0.0070 | 0.0710 ± 0.0167 | -0.0993 ± 0.1213 |
| Pre-sleep | XGBoost | 0.0631 ± 0.0105 | 0.0873 ± 0.0162 | -0.7248 ± 0.4752 |
| Overnight | Mean baseline | 0.0410 ± 0.0070 | 0.0632 ± 0.0147 | -0.0545 ± 0.0588 |
| Overnight | ElasticNetCV | 0.0422 ± 0.0074 | 0.0648 ± 0.0136 | -0.1214 ± 0.1551 |
| Overnight | XGBoost | 0.0509 ± 0.0070 | 0.0739 ± 0.0149 | -0.5240 ± 0.6089 |

ElasticNetCV selected strong regularization and shrank the core HRV coefficients to approximately zero in the overnight model, indicating little stable linear signal under participant-wise evaluation.


### Core vs. Expanded Features

Adding more HRV metrics did not materially improve performance.

For ElasticNetCV, the expanded feature set produced little or no improvement. For XGBoost, expanded features slightly improved pre-sleep MAE but worsened overnight generalization.

This suggests that additional HRV metrics were largely redundant or too noisy to add stable cross-participant predictive value.


## 7. Sensitivity Analysis

Stricter feasibility thresholds were also tested:

- **Pre-sleep:** ≥10 valid segments
- **Overnight:** ≥15 valid segments

The qualitative result was unchanged: ElasticNetCV remained close to the mean-prediction baseline and XGBoost performed worse.

This supports the conclusion that weak predictive performance was not simply an artifact of permissive segment thresholds.


## 8. Symptom Profile Clustering

Participant symptom profiles were created from:

- PHQ-9
- GAD-7
- ISI

Scores were standardized and clustered using **K-means with k=3**.


In [ ]:
survey = pd.read_csv("survey.csv")

cluster_df = (
    survey[["deviceId", "PHQ9_1", "GAD7_1", "ISI_1"]]
    .dropna()
    .copy()
)

X = cluster_df[["PHQ9_1", "GAD7_1", "ISI_1"]]
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(
    n_clusters=3,
    random_state=0,
    n_init=20
)

cluster_df["cluster"] = kmeans.fit_predict(X_scaled)

cluster_summary = (
    cluster_df
    .groupby("cluster")[["PHQ9_1", "GAD7_1", "ISI_1"]]
    .mean()
    .round(2)
)

cluster_summary


### Cluster Summary

| Cluster | PHQ-9 mean | GAD-7 mean | ISI mean | Participants |
|---|---:|---:|---:|---:|
| 0 | 1.11 | 1.04 | 4.41 | 27 |
| 1 | 5.83 | 4.28 | 9.83 | 18 |
| 2 | 11.00 | 10.75 | 16.75 | 4 |

These clusters were interpreted as progressively higher symptom burden.

Participant-level mean sleep efficiency also differed descriptively across profiles. In the overnight dataset:

- Low symptom: **0.9510**
- Moderate symptom: **0.9163**
- High symptom: **0.8840**

The overnight Kruskal–Wallis test produced **p = 0.0506**, while the pre-sleep comparison produced **p = 0.1061**.


## 9. Key Takeaways

- Built an end-to-end workflow for real-world wearable data, including QC, window construction, feature engineering, modeling, and clustering.
- Used **participant-wise GroupKFold** to avoid data leakage between train and test sets.
- Found that the **overnight window was substantially more feasible** than the 2-hour pre-sleep window.
- HRV-only models showed **limited cross-participant predictive signal** for sleep efficiency.
- ElasticNetCV generalized more consistently than XGBoost, but generally did not outperform a simple mean-prediction baseline.
- Expanded HRV features did not provide meaningful improvement.
- Symptom clustering suggested a descriptive gradient in sleep efficiency across symptom burden groups.


## 10. Limitations and Next Steps

### Limitations
- Sleep efficiency was right-skewed with limited variation.
- HRV availability was sparse in the fixed pre-sleep window.
- Free-living PPG-derived HRV is subject to motion and measurement noise.
- Participant heterogeneity may weaken a single cross-person prediction model.

### Potential next steps
- Add activity, motion, light, and sleep-timing features.
- Evaluate other sleep outcomes such as WASO or sleep latency.
- Explore participant-specific or hierarchical models.
- Test whether multimodal features improve generalization beyond HRV alone.


## Skills Demonstrated

**Python · pandas · NumPy · scikit-learn · XGBoost · Data Cleaning · Feature Engineering · Grouped Cross-Validation · Regression · Clustering · Model Evaluation · Wearable Health Data Analysis**

---

For the full exploratory workflow and intermediate analyses, see the original analysis notebook in this repository.
